In this notebook we detect trees and extract their heights.

Inputs: NDVI, DTM and DEM rasters

Processing steps (for every DSM file):
- calculate height of objects: DEM-DTM
- make a tree mask considering NDVI > 0.2 and height >2m
- mask the height raster: get a new height raster with only trees height
- reduce raster noise: apply max, median and mean filters
- simplify raster: make "blocks" every 1.5m
- extract the polygons for each different height of the raster
- saves a gdf with the results: 50 gdf cover all the city

Cleaning steps (for every gdf saved):
- remove polygons with height 0
- remove polygons with area <= 1.5m
- assigns CRS 4326 (neccesary for using pybdshadow library)
- explode multypoligons if there are
- simplify and smooth geometries

note: during all the steps, the temporary and previous files are removed to avoid memory issues (both disk and ram). Also every 20 iterations there is a sleep(30) to avoid memory overload

In [1]:
!pip install geopandas shapely rasterio



[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: C:\Users\guada\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import io
import json
import math
import os
import time

import boto3
import numpy as np
import pandas as pd
from tqdm import tqdm
from scipy.ndimage import maximum_filter, median_filter, uniform_filter

import geopandas as gpd
from shapely.geometry import box, shape, Polygon, MultiPolygon

import rasterio
from rasterio.crs import CRS
from rasterio.features import shapes
from rasterio.warp import reproject, Resampling
from rasterio.windows import from_bounds
import glob


In [3]:
# INPUT FOLDER WITH TREE HEIGHT RASTERS
rasters_folder_path = "../../../../datos-notebooks/2.global-data-bologna/2.2.objects-height/gch_bologna"
trees_vectors_path = "../../../../datos-notebooks/2.global-data-bologna/2.5.MRT-prediction/2.5.1.shadows/trees-vectors/"
clean_trees_vectors_path = "../../../../datos-notebooks/2.global-data-bologna/2.5.MRT-prediction/2.5.1.shadows/trees-vectors-clean/"

In [4]:
# CONFIGURATION

height_list = sorted(glob.glob(os.path.join(rasters_folder_path, "*.tif")))
os.makedirs(trees_vectors_path, exist_ok=True)
num_parts = 20 # number of parts to divide the trees vector in order to avoid memory issues
processed_file_list = "processed_files.json" # to follow the processing progress

In [ ]:
# processes trees height raster to extract polygons with height information - 1h aprox

# LOAD LIST OF ALREADY PROCESSED FILES
if os.path.exists(processed_file_list):
    with open(processed_file_list, "r") as f:
        processed_files = set(json.load(f))
else:
    processed_files = set()

total_files = len(height_list)
group_size = math.ceil(total_files / num_parts)

for i, raster_path in enumerate(tqdm(height_list)):

    filename = os.path.basename(raster_path)

    if filename in processed_files:
        continue

    try:
        # Load height raster
        with rasterio.open(raster_path) as src:
            height = src.read(1)
            transform = src.transform
            crs = src.crs

        # Simple tree mask (height threshold)
        height = np.where(height > 2, height, 0)

        # Apply filters
        h1 = maximum_filter(height, size=3)
        h2 = median_filter(h1, size=7)
        h3 = uniform_filter(h2, size=2)
        height = np.round(h3 / 1.5) * 1.5

        del h1, h2, h3

        # Extract polygons
        results = [
            {
                "height": int(v),
                "geometry": shape(s),
                "filename": filename
            }
            for s, v in shapes(
                height.astype(np.int16),
                transform=transform
            )
            if v > 0
        ]

        gdf = gpd.GeoDataFrame(
            results,
            geometry="geometry",
            crs=crs
        )

        # Save partial results
        group_id = i // group_size
        part_file = os.path.join(trees_vectors_path, f"part{group_id+1}.json")

        if os.path.exists(part_file):
            gdf_old = gpd.read_file(part_file)
            gdf = pd.concat([gdf_old, gdf], ignore_index=True)

        gdf.to_file(part_file, driver="GeoJSON")

        # Register processed file
        processed_files.add(filename)
        with open(processed_file_list, "w") as f:
            json.dump(sorted(processed_files), f)

        del height, gdf, results

        if i % 20 == 0 and i != 0:
            time.sleep(30)

    except Exception as e:
        print(f"Error processing {filename}: {e}")
        continue

os.remove('processed_files.json')


In [ ]:
p1 = "../../../../datos-notebooks/2.global-data-bologna/2.5.MRT-prediction/2.5.1.shadows/trees-vectors/part1.json"

gdf = gpd.read_file(p1)
print(gdf.crs)



EPSG:32632


In [ ]:
# THIS BLOCK TAKES AROUND 6H TO FINISH. 
# If you have the partial_results.tar.gz file you can jump to the next one. 

# This script processes DSM and DTM rasters to extract tree polygons using height and NDVI masks.
# It works with a large number of files, so it divides the workload into 50 parts to avoid memory issues.

'''
dsm_list = s3.list_objects_v2(Bucket=bucket_name, Prefix=path_dsm)['Contents']

# CONFIGURATION
output_dir = "partial_results"
os.makedirs(output_dir, exist_ok=True)
processed_file_list = "processed_files.json"

# LOAD LIST OF ALREADY PROCESSED FILES
if os.path.exists(processed_file_list):
    with open(processed_file_list, 'r') as f:
        processed_files = set(json.load(f))
else:
    processed_files = set()

# CALCULATE TOTAL FILES AND GROUP SIZE
total_files = len(dsm_list)
num_parts = 50
group_size = math.ceil(total_files / num_parts)

for i, obj in enumerate(tqdm(dsm_list)):
    
    filename = obj['Key'].split('/')[-1]
  
    # Skip non-ASC and already processed files
    if not filename.endswith('.ASC') or filename in processed_files:
        continue

    try:
        # Load DSM raster
        dsm_data = s3.get_object(Bucket=bucket_name, Key=path_dsm + filename)
        with rasterio.open(io.BytesIO(dsm_data['Body'].read())) as src_dsm:
            dsm_array = src_dsm.read(1)
            dsm_transform = src_dsm.transform
            dsm_crs = src_dsm.crs or CRS.from_epsg(7791)
            dsm_bounds = box(*src_dsm.bounds)

        # Load DTM raster
        dtm_data = s3.get_object(Bucket=bucket_name, Key=path_dtm + filename)
        with rasterio.open(io.BytesIO(dtm_data['Body'].read())) as src_dtm:
            dtm_array = src_dtm.read(1)

        # Calculate object height 
        height = np.round(dsm_array - dtm_array)
        del dtm_array # make some space in RAM

        # Get area and geometry of DSM tile
        dsm_poly = box(*src_dsm.bounds)
        area_dsm = dsm_poly.area
        
        # Filter NDVI rasters that intersect with current DSM
        candidates = gdf_ndvi_bounds[gdf_ndvi_bounds.intersects(dsm_poly)]
        matching_ndvi = None
        
        for idx, row in candidates.iterrows():
            ndvi_poly = row.geometry
            intersection_area = dsm_poly.intersection(ndvi_poly).area
            if intersection_area / area_dsm > 0.5: #because there ir a 30% overlapping and can consider the wrong ndvi image
                matching_ndvi = row["local_path"]
                break
        
        if matching_ndvi is None:
            print(f"No matching NDVI found with >50% overlap for {filename}")
            continue

        # Load NDVI raster and align it with DSM
        with rasterio.open(matching_ndvi) as src_ndvi:
            dsm_bounds = rasterio.coords.BoundingBox(*src_dsm.bounds)
            if src_ndvi.crs != dsm_crs:
                ndvi_array = np.empty(dsm_array.shape, dtype=np.float32)
                reproject(
                    source=rasterio.band(src_ndvi, 1),
                    destination=ndvi_array,
                    src_transform=src_ndvi.transform,
                    src_crs=src_ndvi.crs,
                    dst_transform=dsm_transform,
                    dst_crs=dsm_crs,
                    dst_nodata=0,
                    resampling=Resampling.nearest,
                    num_threads=2
                )
            else:
                window = from_bounds(*dsm_bounds, transform=src_ndvi.transform)
                ndvi_array = src_ndvi.read(1, window=window, out_shape=dsm_array.shape, resampling=Resampling.nearest)

            # Create binary mask for tree candidates
            tree_mask1 = ((ndvi_array != 0)).astype(np.uint8)
            tree_mask2 = ((height > 2)).astype(np.uint8)
            tree_mask = tree_mask1 * tree_mask2

        height = height * tree_mask 
        del ndvi_array, tree_mask1, tree_mask2, tree_mask, dsm_array # make some space in RAM

        # Apply filters to the height array
        height_filtered_1 = maximum_filter(height, size=3)
        height_filtered_2 = median_filter(height_filtered_1, size=7)
        height_filtered_3 = uniform_filter(height_filtered_2, size=2)
        height = np.round(height_filtered_3 / 1.5) * 1.5

        del height_filtered_1, height_filtered_2,  height_filtered_3 # make some space in RAM

        # Extract polygons from the filtered height raster
        results = [
            {'height': int(v), 'geometry': shape(s), 'filename': filename}
            for s, v in shapes(height.astype(np.int16), transform=dsm_transform)
        ]

        # Create GeoDataFrame
        gdf = gpd.GeoDataFrame(results, geometry='geometry', crs='EPSG:4326')

        # Save to part file inside the output directory
        group_id = i // group_size
        part_file = os.path.join(output_dir, f"part{group_id+1}.json")

        if os.path.exists(part_file):
            with open(part_file, 'r') as f:
                old_data = json.load(f)
            gdf_old = gpd.GeoDataFrame.from_features(old_data, crs='EPSG:4326')
            gdf = pd.concat([gdf_old, gdf], ignore_index=True)

        gdf.to_file(part_file, driver='GeoJSON')

        # Register file as processed
        processed_files.add(filename)
        with open(processed_file_list, 'w') as f:
            json.dump(sorted(processed_files), f)

        del height, results, gdf

        # pause to avoid memory overload
        if i % 20 == 0 and i != 0:
            print(f"Pause at iteration {i}...")
            time.sleep(30)

    except Exception as e:
        print(f"Error processing {filename}: {e}")
        continue
'''

In [5]:
# functions for smoothing results

def chaikin_smoothing(coords, refinements=2):
    for _ in range(refinements):
        new_coords = []
        for i in range(len(coords) - 1):
            p0 = np.array(coords[i])
            p1 = np.array(coords[i + 1])
            Q = 0.75 * p0 + 0.25 * p1
            R = 0.25 * p0 + 0.75 * p1
            new_coords.extend([Q, R])
        new_coords.append(new_coords[0])  # cerrar el anillo
        coords = new_coords
    return coords

def smooth_polygon(poly, refinements=2):
    if isinstance(poly, Polygon):
        smoothed = Polygon(chaikin_smoothing(list(poly.exterior.coords), refinements))
        return smoothed
    elif isinstance(poly, MultiPolygon):
        return MultiPolygon([smooth_polygon(p, refinements) for p in poly.geoms])
    else:
        return poly


In [ ]:
# Cleaning process
# Process each partial file individually and save the results in clean_results
os.makedirs(clean_trees_vectors_path, exist_ok=True)

for j in tqdm(range(num_parts)):
    part_file = os.path.join(trees_vectors_path, f"part{j+1}.json")
    if not os.path.exists(part_file):
        continue

    gdf = gpd.read_file(part_file)
    gdf = gdf.set_crs(32632, allow_override=True)
    gdf = gdf[gdf['height'] != 0]
    gdf = gdf.explode(index_parts=False)
    time.sleep(5)
    gdf['geometry'] = gdf['geometry'].apply(lambda geom: smooth_polygon(geom, refinements=5))
    gdf['geometry'] = gdf['geometry'].simplify(tolerance=0.15, preserve_topology=True)
    gdf = gdf[gdf.geometry.notnull()]
    gdf = gdf[gdf.geometry.area >= 1.5]
    gdf['geometry'] = gdf['geometry'].buffer(0)
    gdf = gdf.to_crs("EPSG:4326")
    clean_path = os.path.join(clean_trees_vectors_path, f"part{j+1}_clean.geojson")
    gdf.to_file(clean_path, driver="GeoJSON")

    #os.remove(part_file) #removes the input file to make some space in disk

  5%|▌         | 1/20 [1:30:22<28:37:00, 5422.13s/it]